# Lambda2 x Depth Tuning for CSET

`lambda2-tuning.ipynb` swept `lambda2` for `CSET` and `MCE` jointly, at a fixed,
hardcoded `depth=0.8` (the shared `wCSET`/`CSET`/`MCE` base class default -- never
itself tuned). `harmonic-prior-tuning.ipynb` ran into exactly this same "fixed,
arbitrary depth" problem for `wCSET`'s `beta`, and found that `beta` and `depth`
interact rather than being independently tunable -- the same risk applies here:
tuning `lambda2` alone at a fixed `depth` could land on a `lambda2` that's only
optimal for that one `depth`.

This notebook narrows to `CSET` only (dropping `MCE` -- whether the same joint
tuning matters for `MCE` is a separate follow-up, not this notebook's scope) and
sweeps `lambda2` **and** `depth` jointly, same `beta x depth`-style joint grid as
`harmonic-prior-tuning.ipynb`, using the same subject/run setup and `MetricSuite`
conventions as `lambda2-tuning.ipynb`.

**Restricted to hrp/pp only, not a full time-window correlation metric**, same
reason as `lambda2-tuning.ipynb`: `CSET`'s cost scales with the number of
timepoints in the data passed to `.solve()` (confirmed there: ~3.7s/solve on a
2-timepoint `phase_data` array), so this stays restricted to the two solved
timepoints (half-rising-phase, peak-phase) rather than a full cropped window.

**Grids**:
- `lambda2_values = [0.01, 0.03, 0.1, 0.3, 1.0]` -- unchanged from `lambda2-tuning.ipynb`
  (5 log-spaced values centered on the `0.1` codebase default), since this notebook
  only adds a second dimension, not a wider `lambda2` range. (`lambda2-tuning.ipynb`'s
  own recommendation flagged `1.0` as the sweep's edge with the true optimum
  possibly higher -- still true here, not addressed by this notebook.)
- `depths = [0, 0.2, 0.5, 0.8, 1.0]` -- same grid as `harmonic-prior-tuning.ipynb`'s
  joint sweep, same rationale (0 = no weighting, 0.8 = class default, the rest for
  range).

Projected cost: 10 runs x 5 lambda2 x 5 depths x ~3.7s/solve (CSET only, no MCE) ~
~15-16 minutes.


> **Project copy** of `mindmap/tutorials/analysis/inverse/cset-lambda2-depth-tuning.ipynb`, not yet re-run.
> Changes: absolute path to the tutorial config, and the runs are the 50 tuning regions in
> `results/simf/tuning/tuning-regions.csv` (simf10 regions outside Study I's probe set, chosen by
> `gbf_cutoff_tuning.ipynb`) instead of runs 1–10. Outputs are written to `results/simf/tuning/`.
> Everything else (grid, preprocessing, metrics) is unchanged from the original.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import mindmap as mm
import mne

from mindmap.analysis.functional.common.metric import find_peak_localization_error
from mindmap.analysis.simulation.engine import Simulator
from mindmap.core.metric import calc_correlation

dm = mm.io.TutoManager("/home/nino/Code/mindmap/tutorials/config.yml")
sub = "simf10"
import pandas as pd
from pathlib import Path
# tuning regions shared with gbf_cutoff_tuning.ipynb: 50 simf10 regions outside Study I's probe set
TUNING = Path("/home/nino/Manuscripts/wCSET/results/simf/tuning")
runs = pd.read_csv(TUNING / "tuning-regions.csv", dtype={"run": str})["run"].tolist()

fwd = dm.load(sub, "fwd", desc="inv")
eig = dm.load(sub, "eig")
eigvec, eigval = eig["vec"], eig["val"]

simulator, _, _ = Simulator.load(dm.roots["sim"]/f"sub-{sub}"/"simulator")
dipole_positions = mm.anatomy.source_space.get_dipole_positions(fwd["src"]) * 1000

tmin, tmax = -0.05, 0.15
lambda2_values = [0.01, 0.03, 0.1, 0.3, 1.0]
depths = [0, 0.2, 0.5, 0.8, 1.0]


## Solve and evaluate across runs x lambda2 x depth (CSET only)

For each run: crop to the response window, locate half-rising-phase/peak-phase on
the cropped `ave`, build a 2-timepoint `phase_data` array (not the full crop) --
that's what actually gets passed to `.solve()`, keeping every solve cheap
regardless of the crop window's own length. Score against that run's own `gold`:
AP (precision-recall) at hrp and pp, spatial correlation (across dipoles) at hrp/pp,
and peak-localization-error at hrp -- same metrics, same construction as
`lambda2-tuning.ipynb`.

Registered into `metrics` under `label=f"lambda2={lambda2}"` (no method prefix needed
now that only `CSET` is being swept) with `depth=depth` carried as an extra per-entry
coordinate tag, same convention `harmonic-prior-tuning.ipynb` uses for `beta`/`depth`.


In [ ]:
metrics = mm.metric.MetricSuite(f"Lambda2 x Depth Tuning {sub} (CSET)")
scalar_rows = []

for run in tqdm(runs, desc="runs"):
    ave = dm.load(sub, "ave", run=run)
    ncov = dm.load(sub, "cov", run=run)
    gold = dm.load(sub, "gold", run=run)
    src_seed = dm.load(sub, "source", run=run)

    data = ave.copy().crop(tmin=tmin, tmax=tmax)
    ground = gold.copy().crop(tmin=tmin, tmax=tmax)
    hrp_idx = mm.analysis.eeg.get_rising_phase(data, ratio=0.5, as_idx=True)[0]
    pp_idx = mm.analysis.eeg.get_rising_phase(data, ratio=1, as_idx=True)[0]
    phase_data = mne.EvokedArray(data.data[:, [hrp_idx, pp_idx]], data.info, tmin=0, nave=ave.nave, kind=ave.kind)

    gold_hrp = ground.data[:, hrp_idx]
    gold_pp = ground.data[:, pp_idx]
    gold_phase = np.stack([gold_hrp, gold_pp], axis=1)   # (n_dipoles, 2), same layout as sol.data
    true_source_mask = src_seed.select_targets(simulator.hm.src, simulator.hm.parc, simulator.hm.ras2tkras)

    def evaluate_and_register(label, sol, depth):
        sol_hrp, sol_pp = sol.data[:, 0], sol.data[:, 1]

        ap_hrp = mm.core.calc_pr_curve(gold_hrp > 0, sol_hrp)["ap"]
        ap_pp = mm.core.calc_pr_curve(gold_pp > 0, sol_pp)["ap"]
        metrics.register("ap_hrp", ap_hrp, Label=label, depth=depth)
        metrics.register("ap_pp", ap_pp, Label=label, depth=depth)

        # spatial correlation (across dipoles) at hrp/pp individually -- axis=0
        # correlates along the dipole axis at each of the 2 timepoints, same
        # "spatial" mode harmonic-prior-tuning.ipynb uses over a full window
        corr = calc_correlation(gold_phase, sol.data, method="pearson", axis=0)
        corr_hrp, corr_pp = corr[0], corr[1]
        metrics.register("spatial_correlation_hrp", corr_hrp, Label=label, depth=depth)
        metrics.register("spatial_correlation_pp", corr_pp, Label=label, depth=depth)

        hit_hrp = find_peak_localization_error(sol_hrp, true_source_mask, dipole_positions)
        metrics.register("localization_error_hrp", hit_hrp["distance_mm"], Label=label, depth=depth)

        return ap_hrp, ap_pp, corr_hrp, corr_pp, hit_hrp["distance_mm"]

    for lambda2 in lambda2_values:
        for depth in depths:
            inv_cset = mm.analysis.inverse.CSET(forward=fwd, noise_cov=ncov, depth=depth, eigenvectors=eigvec)
            sol_cset = inv_cset.solve(phase_data, lambda2=lambda2)
            ap_hrp, ap_pp, corr_hrp, corr_pp, loc_err_hrp = evaluate_and_register(f"lambda2={lambda2}", sol_cset, depth)
            scalar_rows.append(dict(
                run=run, method="cset", lambda2=lambda2, depth=depth,
                ap_hrp=ap_hrp, ap_pp=ap_pp, corr_hrp=corr_hrp, corr_pp=corr_pp, loc_err_hrp=loc_err_hrp,
            ))

scalar_df = pd.DataFrame(scalar_rows)
scalar_df.to_csv(TUNING / "cset_lambda2_depth_tuning_outcomes.tsv", sep="\t", index=False)
print(f"{len(scalar_df)} (run, lambda2, depth) rows.")

## Visualization

Joint `lambda2 x depth` tuning landscape: mean and std, across the 10 runs, for
each scalar metric -- same heatmap-grid approach `harmonic-prior-tuning.ipynb` uses
for its `beta x depth` grid (a box plot per `(lambda2, depth)` cell, as
`lambda2-tuning.ipynb` used for its 1D sweep, would mean 25 boxes per metric --
unreadable at this grid size). No depth-faceted time-course plot here (unlike
`harmonic-prior-tuning.ipynb`): CSET is only solved at 2 timepoints (hrp, pp), not
a full cropped window, so there's no time-series metric to facet.


In [ ]:
scalar_metrics = [
    ("ap_hrp", "RdYlGn"),               # higher is better
    ("ap_pp", "RdYlGn"),                # higher is better
    ("corr_hrp", "RdYlGn"),             # higher is better
    ("corr_pp", "RdYlGn"),              # higher is better
    ("loc_err_hrp", "RdYlGn_r"),        # lower is better -- reversed colormap
]

fig, axes = plt.subplots(2, len(scalar_metrics), figsize=(5 * len(scalar_metrics), 8))
for col, (metric, cmap) in enumerate(scalar_metrics):
    mean_grid = scalar_df.pivot_table(index="depth", columns="lambda2", values=metric, aggfunc="mean")
    std_grid = scalar_df.pivot_table(index="depth", columns="lambda2", values=metric, aggfunc="std")

    sns.heatmap(mean_grid, annot=True, fmt=".2f", cmap=cmap, ax=axes[0, col], cbar_kws={"label": "mean"})
    axes[0, col].set_title(f"{metric} -- mean across runs")

    sns.heatmap(std_grid, annot=True, fmt=".2f", cmap="Greys", ax=axes[1, col], cbar_kws={"label": "std"})
    axes[1, col].set_title(f"{metric} -- std across runs")

fig.suptitle(
    f"Joint lambda2 x depth-weighting tuning -- {sub}, CSET, runs {runs[0]}-{runs[-1]}",
    fontsize=13, fontweight="bold",
)
fig.tight_layout()
fig.savefig(TUNING / "cset_lambda2_depth_tuning.png", dpi=130, bbox_inches="tight")

print("Summary (mean +/- std across the 10 runs):")
scalar_df.groupby(["lambda2", "depth"])[["ap_hrp", "ap_pp", "corr_hrp", "corr_pp", "loc_err_hrp"]].agg(["mean", "std"])
